# RAG Inference Pipeline

In [10]:
import json
import os
import time
import numpy as np
import faiss
import torch
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, AutoModelForCausalLM
from typing import List, Dict, Optional
from tqdm.auto import tqdm

In [11]:
EMBEDDING_DIR = "data"
RESULTS_DIR   = "results/"
os.makedirs(RESULTS_DIR, exist_ok=True)

INPUT_JSON  = "examples/benchmark_unified.json"
OUTPUT_JSON = os.path.join(RESULTS_DIR, "output_results.json")

# -- Retrieval config (best combination from evaluation) --
CHUNK_STRATEGY = "chunks_fixed300_overlap0"
EMBED_MODEL    = "nomic"                       # "minilm" | "nomic"
USE_RERANKER   = True
TOP_K          = 5
TOP_N_RERANK   = 20

# -- Generation config (best from exploration) --
QWEN_MODEL     = "Qwen/Qwen2.5-0.5B-Instruct"
PROMPT_TYPE    = "basic"
MAX_NEW_TOKENS = 64
TEMPERATURE    = 0.0
TOP_P          = 1.0
TOP_K_GEN      = 50

# -- Model registry (local only, no API key needed) --
MODEL_REGISTRY = {
    "minilm": {
        "type": "sentence_transformer",
        "model_name": "all-MiniLM-L6-v2",
    },
    "nomic": {
        "type": "sentence_transformer",
        "model_name": "nomic-ai/nomic-embed-text-v1.5",
        "trust_remote_code": True,
    },
}

# -- Derived paths --
FAISS_PATH    = os.path.join(EMBEDDING_DIR, CHUNK_STRATEGY, f"embedded_{EMBED_MODEL}.faiss")
JSONL_PATH    = os.path.join(EMBEDDING_DIR, CHUNK_STRATEGY, f"embedded_{EMBED_MODEL}.jsonl")
CROSS_ENCODER_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"

print(f"FAISS index    : {FAISS_PATH}")
print(f"Corpus JSONL   : {JSONL_PATH}")
print(f"Embed model    : {EMBED_MODEL} -> {MODEL_REGISTRY[EMBED_MODEL]['model_name']}")
print(f"Reranker       : {'ON' if USE_RERANKER else 'OFF'}")
print(f"Prompt type    : {PROMPT_TYPE}")
print(f"Generation     : temp={TEMPERATURE}, top_p={TOP_P}, top_k={TOP_K_GEN}, max_tokens={MAX_NEW_TOKENS}")


FAISS index    : data\chunks_fixed300_overlap0\embedded_nomic.faiss
Corpus JSONL   : data\chunks_fixed300_overlap0\embedded_nomic.jsonl
Embed model    : nomic -> nomic-ai/nomic-embed-text-v1.5
Reranker       : ON
Prompt type    : basic
Generation     : temp=0.0, top_p=1.0, top_k=50, max_tokens=64


In [12]:

_encoder_cache = {}

def get_query_encoder(model_tag: str):
    """
    Returns encode_fn(texts) -> np.ndarray (n, dim), normalised.
    Uses the correct model for the given tag.
    """
    if model_tag in _encoder_cache:
        return _encoder_cache[model_tag]

    config = MODEL_REGISTRY[model_tag]

    if config["type"] == "sentence_transformer":
        kwargs = {}
        if config.get("trust_remote_code"):
            kwargs["trust_remote_code"] = True
        st_model = SentenceTransformer(config["model_name"], **kwargs)

        def encode_fn(texts):
            return st_model.encode(
                texts, normalize_embeddings=True, show_progress_bar=True
            ).astype(np.float32)

    else:
        raise ValueError(f"Unknown model type: {config['type']}")

    _encoder_cache[model_tag] = encode_fn
    print(f"Query encoder loaded: {config['model_name']}")
    return encode_fn

## 1. Load Retrieval Resources

In [13]:
# Load FAISS index + corpus
index = faiss.read_index(FAISS_PATH)
with open(JSONL_PATH, "r", encoding="utf-8") as f:
    corpus_lines = f.readlines()
assert len(corpus_lines) == index.ntotal
print(f"FAISS index: {index.ntotal} vectors, dim={index.d}")

# Load query encoder (matched to FAISS index)
encode_fn = get_query_encoder(EMBED_MODEL)

# Optional cross-encoder
cross_encoder = None
if USE_RERANKER:
    cross_encoder = CrossEncoder(CROSS_ENCODER_MODEL)
    print(f"Cross-encoder loaded: {CROSS_ENCODER_MODEL}")

FAISS index: 1838 vectors, dim=768


<All keys matched successfully>


Query encoder loaded: nomic-ai/nomic-embed-text-v1.5


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Cross-encoder loaded: cross-encoder/ms-marco-MiniLM-L-6-v2


In [14]:
def retrieve(queries: List[str], top_n: int) -> List[List[Dict]]:
    """
    Bi-encoder retrieval. Takes a list of query strings,
    returns list of list of {rank, chunk_id, score, text}.
    """
    query_embs = encode_fn(queries)
    scores_all, indices_all = index.search(query_embs, top_n)

    all_results = []
    for scores, indices in zip(scores_all, indices_all):
        retrieved = []
        for rank, (score, idx) in enumerate(zip(scores, indices), start=1):
            if idx == -1:
                continue
            chunk = json.loads(corpus_lines[idx])
            retrieved.append({
                "rank": rank,
                "chunk_id": chunk["chunk_id"],
                "score": round(float(score), 6),
                "text": chunk["text"],
            })
        all_results.append(retrieved)
    return all_results


def rerank_results(query: str, candidates: List[Dict], top_k: int) -> List[Dict]:
    """Cross-encoder reranking on a single query's candidates."""
    if not candidates or not cross_encoder:
        return candidates[:top_k]

    pairs = [(query, c["text"]) for c in candidates]
    ce_scores = cross_encoder.predict(pairs)
    scored = sorted(zip(ce_scores, candidates), key=lambda x: x[0], reverse=True)

    reranked = []
    for new_rank, (ce_score, cand) in enumerate(scored[:top_k], start=1):
        reranked.append({
            **cand,
            "rank": new_rank,
            "rerank_score": round(float(ce_score), 6),
        })
    return reranked

## 2. Load Qwen2.5 Generator

In [15]:
tokenizer = AutoTokenizer.from_pretrained(QWEN_MODEL, trust_remote_code=True)
qwen_model = AutoModelForCausalLM.from_pretrained(
    QWEN_MODEL,
    dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True,
)
print(f"Generator: {QWEN_MODEL}")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Generator: Qwen/Qwen2.5-0.5B-Instruct


## 3. Prompt Templates & Generation

In [16]:
MAX_CONTEXT_CHARS = 2048

def build_context(retrieved: List[Dict]) -> str:
    """Assemble retrieved chunks into a context string."""
    if not retrieved:
        return "(No relevant context found. If needed, state that the answer is not available in the provided context.)"
    parts = []
    total = 0
    for r in retrieved:
        block = f"[{r['chunk_id']}] {r['text']}"
        if total + len(block) > MAX_CONTEXT_CHARS:
            parts.append(block[:MAX_CONTEXT_CHARS - total] + "...")
            break
        parts.append(block)
        total += len(block)
    return "\n\n".join(parts)


PROMPT_TEMPLATES = {
    "basic": (
    "Answer the question using only the context below. "
    "Give a short factual answer in one or two sentences. "
    "Do not add extra background. If the answer is not in the context, say you cannot answer from the provided context.\n\n"
    "Context:\n{context}\n\n"
    "Question: {query}\n\nAnswer:"
    ),

    "enhanced": (
        "You are a culinary expert. Answer the question using ONLY the context provided. "
        "If the context does not contain the answer, say \"Not enough information.\"\n\n"
        "Context:\n{context}\n\n"
        "Question: {query}\n"
        "Answer:"
    ),
    "cot": (
        "Based on the context, answer the question. "
        "Quote the key evidence first, then give a short answer.\n\n"
        "Context:\n{context}\n\n"
        "Question: {query}\n"
        "Evidence and answer:"
    ),
    "few_shot": (
        "Context: Sushi is a Japanese dish of vinegared rice with raw fish.\n"
        "Question: What is sushi?\n"
        "Answer: Sushi is a Japanese dish made of vinegared rice with raw fish.\n\n"
        "Context: Kimchi is a Korean fermented vegetable dish, usually made with napa cabbage.\n"
        "Question: What is kimchi made from?\n"
        "Answer: Kimchi is made from fermented napa cabbage.\n\n"
        "Context:\n{context}\n\n"
        "Question: {query}\n"
        "Answer:"
    ),
}


def generate_response(
    context: str,
    query: str,
    prompt_type: str = "basic",
    max_new_tokens: int = 512,
    temperature: float = 0.7,
    top_p: float = 0.9,
    top_k: int = 50,
) -> str:
    """Build prompt from template, run Qwen2.5 generation, return response text."""
    template = PROMPT_TEMPLATES.get(prompt_type, PROMPT_TEMPLATES["basic"])
    user_prompt = template.format(context=context, query=query)

    messages = [{"role": "user", "content": user_prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(qwen_model.device)

    # Use greedy decoding when temperature is near zero
    do_sample = temperature > 0.01

    gen_kwargs = dict(
        **inputs,
        max_new_tokens=max_new_tokens,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=do_sample,
    )
    if do_sample:
        gen_kwargs.update(temperature=temperature, top_p=top_p, top_k=top_k)

    with torch.no_grad():
        out = qwen_model.generate(**gen_kwargs)
    gen_ids = out[:, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(gen_ids[0], skip_special_tokens=True).strip()

## 4. Run Pipeline

Supported input formats:
- `{"queries": [{"query_id": "...", "query": "..."}, ...]}`
- `{"sources": [{"source": "...", "questions": [{"question": "...", "answer": "..."}, ...]}]}`

If `query_id` is missing, the notebook auto-generates sequential ids such as `q_001`, `q_002`, ...

Output format: `{"results": [{"query_id", "query", "response", "retrieved_context"}, ...]}`

In [17]:
def load_queries(input_path: str) -> List[Dict]:
    """Load queries from either TA format or benchmark/source-question format."""
    with open(input_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    if "queries" in data:
        queries = []
        for idx, item in enumerate(data.get("queries", []), start=1):
            query_text = (item.get("query") or item.get("question") or "").strip()
            if not query_text:
                continue
            query_id = str(item.get("query_id") or f"q_{idx:03d}")
            queries.append({
                "query_id": query_id,
                "query": query_text,
            })
        return queries

    if "sources" in data:
        queries = []
        idx = 1
        for source_block in data.get("sources", []):
            for item in source_block.get("questions", []):
                query_text = (item.get("question") or item.get("query") or "").strip()
                if not query_text:
                    continue
                query_id = str(item.get("query_id") or f"q_{idx:03d}")
                queries.append({
                    "query_id": query_id,
                    "query": query_text,
                })
                idx += 1
        return queries

    raise ValueError(
        "Unsupported input format. Expected either 'queries' or 'sources' at the top level."
    )


def run_pipeline(
    input_path: str,
    output_path: str,
    prompt_type: str = PROMPT_TYPE,
    max_new_tokens: int = 256,
    temperature: float = 0.7,
    top_p: float = 0.9,
    top_k: int = 50,
) -> Dict:
    """Full RAG pipeline: load queries -> retrieve -> generate -> save."""
    queries_data = load_queries(input_path)

    if not queries_data:
        print("No queries found.")
        return {"results": []}

    query_texts = [q["query"] for q in queries_data]
    print(f"Processing {len(query_texts)} queries...")

    # -- Retrieval --
    t0 = time.time()
    top_n = TOP_N_RERANK if USE_RERANKER else TOP_K
    all_retrieved = retrieve(query_texts, top_n)

    if USE_RERANKER:
        all_retrieved = [
            rerank_results(q, cands, TOP_K)
            for q, cands in zip(query_texts, all_retrieved)
        ]
    print(f"Retrieval done: {time.time()-t0:.2f}s")

    # -- Generation --
    results = []
    for q_data, retrieved in tqdm(
        zip(queries_data, all_retrieved),
        total=len(queries_data),
        desc="Running",
        unit="query",
    ):
        context = build_context(retrieved)
        response = generate_response(
            context, q_data["query"], prompt_type,
            max_new_tokens, temperature, top_p, top_k
        )
        results.append({
            "query_id": q_data["query_id"],
            "query":    q_data["query"],
            "response": response,
            "retrieved_context": [
                {"doc_id": r["chunk_id"], "text": r["text"]} for r in retrieved
            ],
        })

    output = {"results": results}
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(output, f, ensure_ascii=False, indent=2)
    print(f"Saved to {output_path}")
    return output

## 5. Run Pipeline (Read from JSON file)

Point `INPUT_JSON` to the test set provided by TA. Run All to produce output.

In [18]:


# Run pipeline with configured params
output = run_pipeline(
    INPUT_JSON, OUTPUT_JSON,
    prompt_type=PROMPT_TYPE,
    max_new_tokens=MAX_NEW_TOKENS,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    top_k=TOP_K_GEN,
)

# Preview first 3 results
for r in output["results"][:3]:
    print(f"\n{'='*60}")
    print(f"Q: {r['query']}")
    print(f"A: {r['response'][:200]}...")
    print(f"Sources: {[c['doc_id'] for c in r['retrieved_context']]}")


Processing 72 queries...


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Retrieval done: 90.15s


Running:   0%|          | 0/72 [00:00<?, ?query/s]

Saved to results/output_results.json

Q: What does the word 'bibimbap' literally mean in Korean?
A: The word 'bibimbap' literally means "mixed rice."...
Sources: ['wikibooks_0069_chunk_0', 'wiki_0006_chunk_0', 'wiki_0006_chunk_2', 'blog_0022_chunk_0', 'wiki_0006_chunk_7']

Q: What does the Japanese term 'nuuhara' mean?
A: The Japanese term 'nuuhara' means 'thin' or 'thin sauce.'...
Sources: ['wiki_0049_chunk_22', 'wiki_0236_chunk_31', 'wiki_0082_chunk_5', 'wiki_0069_chunk_5', 'wiki_0089_chunk_2']

Q: What is the defining component of all modern sushi?
A: The defining component of all modern sushi is the vinegared rice, also known as shari (しゃり) or sumeshi (酢飯)....
Sources: ['wiki_0259_chunk_0', 'wiki_0241_chunk_0', 'wiki_0259_chunk_7', 'wiki_0259_chunk_1', 'wiki_0259_chunk_5']
